# Convolutional Neural Networks — Chest X-Ray Pneumonia Detection

---

## Copyright and License

This Jupyter Notebook is provided for educational and research purposes.

## Disclaimer

This notebook is provided "as is", without warranty of any kind, express or implied.
All analyses and interpretations are for educational and research purposes only.
This is **not** a medical diagnostic tool.

## Dataset Note

**Dataset:** Chest X-Ray Images (Pneumonia)
**Source:** https://www.kaggle.com/datasets/paultimothymooney/chest-xray-pneumonia
**Locally under:** `../datasets/chest_xray/`
**License:** CC BY 4.0
**Citation:** Kermany et al., Cell 2018. http://www.cell.com/cell/fulltext/S0092-8674(18)30154-5

---

## Abstract

This notebook introduces **Convolutional Neural Networks (CNNs)** and applies them to binary medical image classification: detecting **Pneumonia** vs **Normal** chest X-rays. Building on the MLP foundations from the perceptron notebook, we motivate why fully-connected networks fail on images, introduce the core CNN operations (convolution, pooling, feature maps), and implement two models: a **custom CNN from scratch** and a **transfer learning** approach using a pretrained ResNet-18. Models are evaluated with accuracy, classification report, confusion matrix, and ROC-AUC.

---

## Executive Summary

The Chest X-Ray dataset contains 5,863 JPEG images across two classes (Normal / Pneumonia) split into train, validation, and test sets. A custom 3-block CNN is trained from scratch, then compared against a fine-tuned ResNet-18. Class imbalance is addressed via WeightedRandomSampler. Both models are evaluated on the held-out test set.

---

## Dataset Relevance

Chest X-ray classification is a real-world binary image classification problem with clinical significance. The dataset is moderately imbalanced (more Pneumonia than Normal samples), making it a good testbed for CNNs, data augmentation, and transfer learning in a medical imaging context.

---

## Table of Contents

1. [Import Libraries](#1-import-libraries)
2. [CNN Theory — From MLP to Convolutions](#2-cnn-theory--from-mlp-to-convolutions)
3. [Load & Explore Dataset](#3-load--explore-dataset)
4. [Data Augmentation & DataLoaders](#4-data-augmentation--dataloaders)
5. [Custom CNN from Scratch](#5-custom-cnn-from-scratch)
6. [Training the Custom CNN](#6-training-the-custom-cnn)
7. [Transfer Learning — ResNet-18](#7-transfer-learning--resnet-18)
8. [Training the Fine-Tuned ResNet-18](#8-training-the-fine-tuned-resnet-18)
9. [Model Evaluation & Comparison](#9-model-evaluation--comparison)
10. [Visualising What the CNN Learned](#10-visualising-what-the-cnn-learned)
11. [Conclusions](#11-conclusions)

## 1. Import Libraries

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import datasets, transforms, models
from torchvision.utils import make_grid

from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay,
    roc_auc_score, roc_curve
)

import warnings
warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'PyTorch version : {torch.__version__}')
print(f'Using device    : {device}')

## 2. CNN Theory — From MLP to Convolutions

### Why Not Just Use an MLP on Images?

A 224×224 RGB image has **150,528 input values**. A single fully-connected hidden layer with 512 neurons would require ~77M parameters — just for the first layer. This is:
- Computationally expensive
- Prone to overfitting
- Ignores spatial structure (nearby pixels are related)

### The Convolution Operation

A **convolutional layer** slides a small **kernel** (filter) across the image, computing a dot product at each position. This gives a **feature map** that detects a specific pattern (edge, texture, shape) regardless of where it appears — a property called **translation equivariance**.

Key parameters:

| Parameter | Description |
|---|---|
| `kernel_size` | Size of the filter (e.g. 3×3, 5×5) |
| `stride` | Step size when sliding the kernel |
| `padding` | Zeros added around the border to control output size |
| `out_channels` | Number of filters (= number of feature maps) |

Output spatial size: $\lfloor \frac{H + 2P - K}{S} \rfloor + 1$

### Pooling

**Max pooling** downsamples feature maps by taking the maximum value in each window, reducing spatial dimensions and providing **translation invariance**.

### Typical CNN Architecture

```
Input Image
  -> [Conv -> BN -> ReLU -> MaxPool]  x N   (feature extraction)
  -> Flatten / AdaptiveAvgPool
  -> [Linear -> ReLU -> Dropout]  x M       (classification head)
  -> Linear -> Sigmoid
```

Early layers learn low-level features (edges, colours); deeper layers learn high-level concepts (shapes, objects).

In [ ]:
# Visualise the convolution operation on a synthetic image
img = torch.zeros(1, 1, 8, 8)
img[0, 0, ::2, ::2] = 1.0
img[0, 0, 1::2, 1::2] = 1.0

kernel_h = torch.tensor([[-1,-1,-1],[0,0,0],[1,1,1]], dtype=torch.float32).view(1,1,3,3)
kernel_v = torch.tensor([[-1,0,1],[-1,0,1],[-1,0,1]], dtype=torch.float32).view(1,1,3,3)

out_h = F.conv2d(img, kernel_h, padding=1)
out_v = F.conv2d(img, kernel_v, padding=1)

fig, axes = plt.subplots(1, 3, figsize=(10, 3))
for ax, data, title in zip(axes,
                            [img[0,0], out_h[0,0], out_v[0,0]],
                            ['Input Image', 'Horizontal Edge Filter', 'Vertical Edge Filter']):
    ax.imshow(data.numpy(), cmap='gray')
    ax.set_title(title)
    ax.axis('off')
plt.suptitle('Convolution: Edge Detection Kernels', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Demonstrate how spatial dimensions change through Conv + Pool
demo = torch.randn(1, 3, 224, 224)
conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
pool  = nn.MaxPool2d(2, 2)
conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)

x = demo
print(f'Input          : {tuple(x.shape)}')
x = conv1(x); print(f'After Conv1    : {tuple(x.shape)}  (32 feature maps, same H/W)')
x = pool(x);  print(f'After MaxPool  : {tuple(x.shape)}  (H/W halved)')
x = conv2(x); print(f'After Conv2    : {tuple(x.shape)}  (64 feature maps)')
x = pool(x);  print(f'After MaxPool  : {tuple(x.shape)}  (H/W halved again)')
print(f'Flattened      : {x.view(1,-1).shape[1]:,} values  (vs 150,528 raw pixels)')

## 3. Load & Explore Dataset

**Setup:** Download the dataset from Kaggle and place it at `../datasets/chest_xray/`.
Expected structure:
```
datasets/chest_xray/
├── train/
│   ├── NORMAL/
│   └── PNEUMONIA/
├── val/
│   ├── NORMAL/
│   └── PNEUMONIA/
└── test/
    ├── NORMAL/
    └── PNEUMONIA/
```
Download: `kaggle datasets download -d paultimothymooney/chest-xray-pneumonia`

In [ ]:
DATA_DIR = Path('../datasets/chest_xray')
splits   = ['train', 'val', 'test']
classes  = ['NORMAL', 'PNEUMONIA']

counts = {}
for split in splits:
    counts[split] = {}
    for cls in classes:
        path = DATA_DIR / split / cls
        n = len(list(path.glob('*.jpeg')) + list(path.glob('*.jpg')))
        counts[split][cls] = n

df_counts = pd.DataFrame(counts).T
df_counts['Total'] = df_counts.sum(axis=1)
print(df_counts)
print(f'\nTotal images: {df_counts["Total"].sum()}')

In [ ]:
# Class distribution per split
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
colors = ['steelblue', 'tomato']

for ax, split in zip(axes, splits):
    vals = [counts[split][c] for c in classes]
    bars = ax.bar(classes, vals, color=colors, edgecolor='k', linewidth=0.5)
    ax.set_title(f'{split.capitalize()} split')
    ax.set_ylabel('Count')
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
                str(v), ha='center', fontsize=10)

plt.suptitle('Class Distribution per Split', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Sample images from each class
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
for row, cls in enumerate(classes):
    img_paths = sorted((DATA_DIR / 'train' / cls).glob('*.jpeg'))[:5]
    for col, img_path in enumerate(img_paths):
        img = Image.open(img_path).convert('L')
        axes[row, col].imshow(img, cmap='gray')
        axes[row, col].axis('off')
        if col == 0:
            axes[row, col].set_ylabel(cls, fontsize=11)

plt.suptitle('Sample Chest X-Rays: NORMAL (top) vs PNEUMONIA (bottom)', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Pixel intensity distribution comparison
def sample_pixel_values(split, cls, n=200):
    paths = sorted((DATA_DIR / split / cls).glob('*.jpeg'))[:n]
    pixels = []
    for p in paths:
        arr = np.array(Image.open(p).convert('L').resize((64, 64))).flatten()
        pixels.extend(arr.tolist())
    return pixels

fig, ax = plt.subplots(figsize=(9, 4))
for cls, color in zip(classes, colors):
    px = sample_pixel_values('train', cls)
    ax.hist(px, bins=50, alpha=0.5, color=color, label=cls, density=True)
ax.set_title('Pixel Intensity Distribution (train, 200 images per class)')
ax.set_xlabel('Pixel value (0-255)')
ax.set_ylabel('Density')
ax.legend()
plt.tight_layout()
plt.show()

## 4. Data Augmentation & DataLoaders

In [ ]:
IMG_SIZE   = 224
BATCH_SIZE = 32

# ImageNet mean/std — standard even for grayscale converted to 3-channel
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

# Training: augmentation to reduce overfitting
train_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),  # replicate to 3ch for ResNet compatibility
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

# Val / Test: no augmentation
eval_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

train_dataset = datasets.ImageFolder(DATA_DIR / 'train', transform=train_transform)
val_dataset   = datasets.ImageFolder(DATA_DIR / 'val',   transform=eval_transform)
test_dataset  = datasets.ImageFolder(DATA_DIR / 'test',  transform=eval_transform)

print('Classes      :', train_dataset.classes)
print('Class -> idx :', train_dataset.class_to_idx)
print(f'Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}')

In [ ]:
# WeightedRandomSampler to handle class imbalance
train_labels  = [label for _, label in train_dataset.samples]
class_counts  = np.bincount(train_labels)
class_weights = 1.0 / class_counts
sample_weights = [class_weights[l] for l in train_labels]

sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f'Train batches: {len(train_loader)} | Val: {len(val_loader)} | Test: {len(test_loader)}')

In [ ]:
# Visualise a batch of augmented training images
def denormalize(tensor, mean=MEAN, std=STD):
    t = tensor.clone()
    for c, m, s in zip(range(3), mean, std):
        t[c] = t[c] * s + m
    return t.clamp(0, 1)

images, labels = next(iter(train_loader))
grid = make_grid([denormalize(img) for img in images[:16]], nrow=8, padding=2)

fig, ax = plt.subplots(figsize=(14, 4))
ax.imshow(grid.permute(1, 2, 0).numpy()[:, :, 0], cmap='gray')
ax.axis('off')
ax.set_title('Augmented Training Batch — ' + '  |  '.join(
    [train_dataset.classes[l] for l in labels[:8].tolist()]), fontsize=9)
plt.tight_layout()
plt.show()

## 5. Custom CNN from Scratch

We build a 3-block CNN following the pattern:
`Conv -> BatchNorm -> ReLU -> MaxPool`
followed by a fully-connected classification head with Dropout.

**Batch Normalisation** normalises activations within each mini-batch, stabilising training and allowing higher learning rates.
**Dropout** randomly zeros neurons during training, acting as regularisation to reduce overfitting.

In [ ]:
class CustomCNN(nn.Module):
    """3-block CNN for binary image classification."""

    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            # Block 1: 3 -> 32 channels, 224 -> 112
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            # Block 2: 32 -> 64 channels, 112 -> 56
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            # Block 3: 64 -> 128 channels, 56 -> 28
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((4, 4)),
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(256, 1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


cnn = CustomCNN().to(device)
total     = sum(p.numel() for p in cnn.parameters())
trainable = sum(p.numel() for p in cnn.parameters() if p.requires_grad)
print(cnn)
print(f'\nTotal parameters    : {total:,}')
print(f'Trainable parameters: {trainable:,}')

In [ ]:
# Verify forward pass shape
dummy = torch.randn(2, 3, 224, 224).to(device)
out   = cnn(dummy)
print(f'Input  : {tuple(dummy.shape)}')
print(f'Output : {tuple(out.shape)}  (batch_size x 1 probability)')

## 6. Training the Custom CNN

We use **Binary Cross-Entropy loss** with the **Adam** optimizer and a **ReduceLROnPlateau** scheduler that halves the learning rate when validation loss stops improving for 3 epochs.

In [ ]:
def train_model(model, train_loader, val_loader, epochs=15, lr=1e-3):
    """Training loop with LR scheduler and best-model checkpointing."""
    criterion = nn.BCELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=3, factor=0.5, verbose=True)

    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
    best_val_loss, best_state = float('inf'), None

    for epoch in range(1, epochs + 1):
        model.train()
        t_loss, t_correct = 0.0, 0
        for X_b, y_b in train_loader:
            X_b, y_b = X_b.to(device), y_b.float().unsqueeze(1).to(device)
            optimizer.zero_grad()
            preds = model(X_b)
            loss  = criterion(preds, y_b)
            loss.backward()
            optimizer.step()
            t_loss    += loss.item() * X_b.size(0)
            t_correct += ((preds >= 0.5).float() == y_b).sum().item()

        model.eval()
        v_loss, v_correct = 0.0, 0
        with torch.no_grad():
            for X_b, y_b in val_loader:
                X_b, y_b = X_b.to(device), y_b.float().unsqueeze(1).to(device)
                preds = model(X_b)
                loss  = criterion(preds, y_b)
                v_loss    += loss.item() * X_b.size(0)
                v_correct += ((preds >= 0.5).float() == y_b).sum().item()

        n_tr, n_v = len(train_loader.dataset), len(val_loader.dataset)
        tl, ta = t_loss / n_tr, t_correct / n_tr
        vl, va = v_loss / n_v,  v_correct / n_v

        history['train_loss'].append(tl); history['val_loss'].append(vl)
        history['train_acc'].append(ta);  history['val_acc'].append(va)
        scheduler.step(vl)

        if vl < best_val_loss:
            best_val_loss = vl
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

        print(f'Epoch {epoch:2d}/{epochs} | Train Loss: {tl:.4f}  Acc: {ta:.4f} | Val Loss: {vl:.4f}  Acc: {va:.4f}')

    model.load_state_dict({k: v.to(device) for k, v in best_state.items()})
    print(f'\nRestored best model (val_loss={best_val_loss:.4f})')
    return history

In [ ]:
cnn = CustomCNN().to(device)
torch.manual_seed(SEED)
print('=== Training Custom CNN ===')
history_cnn = train_model(cnn, train_loader, val_loader, epochs=15, lr=1e-3)

In [ ]:
def plot_history(history, title):
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    ep = range(1, len(history['train_loss']) + 1)
    axes[0].plot(ep, history['train_loss'], label='Train')
    axes[0].plot(ep, history['val_loss'],   label='Val', linestyle='--')
    axes[0].set_title('Loss'); axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('BCE Loss'); axes[0].legend()
    axes[1].plot(ep, history['train_acc'], label='Train')
    axes[1].plot(ep, history['val_acc'],   label='Val', linestyle='--')
    axes[1].set_title('Accuracy'); axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy'); axes[1].legend()
    plt.suptitle(title, fontsize=13); plt.tight_layout(); plt.show()

plot_history(history_cnn, 'Custom CNN — Training Curves')

## 7. Transfer Learning — ResNet-18

**Transfer learning** reuses a model pretrained on ImageNet (1.2M images, 1000 classes) as a starting point. The pretrained weights already encode rich visual features (edges, textures, shapes), so we only need to fine-tune on our smaller dataset.

Strategy — **fine-tuning**:
1. Load ResNet-18 with pretrained ImageNet weights
2. Replace the final FC layer with a new binary head
3. Train the entire network end-to-end with a small learning rate

This typically converges faster and achieves higher accuracy than training from scratch.

In [ ]:
def build_resnet18():
    """ResNet-18 with a binary classification head."""
    model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    in_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Linear(in_features, 256),
        nn.ReLU(inplace=True),
        nn.Dropout(0.4),
        nn.Linear(256, 1),
        nn.Sigmoid()
    )
    return model

resnet = build_resnet18().to(device)
total     = sum(p.numel() for p in resnet.parameters())
trainable = sum(p.numel() for p in resnet.parameters() if p.requires_grad)
print(f'Total parameters    : {total:,}')
print(f'Trainable parameters: {trainable:,}')
print(f'\nModified head:\n{resnet.fc}')

## 8. Training the Fine-Tuned ResNet-18

In [ ]:
resnet = build_resnet18().to(device)
torch.manual_seed(SEED)
print('=== Fine-tuning ResNet-18 ===')
history_resnet = train_model(resnet, train_loader, val_loader, epochs=10, lr=1e-4)

In [ ]:
plot_history(history_resnet, 'ResNet-18 Fine-tuned — Training Curves')

## 9. Model Evaluation & Comparison

In [ ]:
def evaluate_model(model, loader, model_name):
    model.eval()
    all_probs, all_labels = [], []
    with torch.no_grad():
        for X_b, y_b in loader:
            probs = model(X_b.to(device)).cpu().numpy().flatten()
            all_probs.extend(probs.tolist())
            all_labels.extend(y_b.numpy().tolist())

    all_probs  = np.array(all_probs)
    all_labels = np.array(all_labels)
    all_preds  = (all_probs >= 0.5).astype(int)
    acc     = accuracy_score(all_labels, all_preds)
    roc_auc = roc_auc_score(all_labels, all_probs)

    print(f'\n=== {model_name} — Test Set ===')
    print(f'Accuracy : {acc:.4f}')
    print(f'ROC-AUC  : {roc_auc:.4f}')
    print(classification_report(all_labels, all_preds, target_names=test_dataset.classes))
    return all_labels, all_preds, all_probs, acc, roc_auc

y_true_cnn,    y_pred_cnn,    y_prob_cnn,    acc_cnn,    auc_cnn    = evaluate_model(cnn,    test_loader, 'Custom CNN')
y_true_resnet, y_pred_resnet, y_prob_resnet, acc_resnet, auc_resnet = evaluate_model(resnet, test_loader, 'ResNet-18')

In [ ]:
# Confusion matrices
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, preds, title in zip(axes,
                             [y_pred_cnn, y_pred_resnet],
                             ['Custom CNN', 'ResNet-18 (fine-tuned)']):
    cm = confusion_matrix(y_true_cnn, preds)
    ConfusionMatrixDisplay(cm, display_labels=test_dataset.classes).plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(f'{title} — Confusion Matrix')
plt.tight_layout()
plt.show()

In [ ]:
# ROC curves
fig, ax = plt.subplots(figsize=(7, 5))
for probs, labels, name, color in [
    (y_prob_cnn,    y_true_cnn,    f'Custom CNN (AUC={auc_cnn:.3f})',    'steelblue'),
    (y_prob_resnet, y_true_resnet, f'ResNet-18  (AUC={auc_resnet:.3f})', 'tomato'),
]:
    fpr, tpr, _ = roc_curve(labels, probs)
    ax.plot(fpr, tpr, label=name, linewidth=2, color=color)
ax.plot([0,1],[0,1],'k--', linewidth=1, label='Random')
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves — Test Set'); ax.legend()
plt.tight_layout(); plt.show()

In [ ]:
# Summary bar chart
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
model_names = ['Custom CNN', 'ResNet-18']
bar_colors  = ['steelblue', 'tomato']
for ax, metric, values in [
    (axes[0], 'Test Accuracy', [acc_cnn, acc_resnet]),
    (axes[1], 'ROC-AUC',       [auc_cnn, auc_resnet]),
]:
    bars = ax.bar(model_names, values, color=bar_colors, edgecolor='k', linewidth=0.5)
    ax.set_ylim(0.7, 1.02); ax.set_ylabel(metric); ax.set_title(metric)
    for bar, v in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{v:.4f}', ha='center', fontsize=11)
plt.suptitle('Custom CNN vs ResNet-18 — Test Set', fontsize=13)
plt.tight_layout(); plt.show()

## 10. Visualising What the CNN Learned

### Misclassified Images
Inspecting errors helps understand model failure modes — e.g. subtle pneumonia cases or noisy X-rays.

### First-Layer Filters
The first convolutional layer's filters are directly interpretable as image patterns the network looks for.

### Feature Maps
Intermediate activations show which spatial regions each filter responds to.

In [ ]:
# Misclassified test images (best model)
best_model = resnet if auc_resnet >= auc_cnn else cnn
best_name  = 'ResNet-18' if auc_resnet >= auc_cnn else 'Custom CNN'

best_model.eval()
mis_imgs, mis_true, mis_pred = [], [], []
with torch.no_grad():
    for X_b, y_b in test_loader:
        probs = best_model(X_b.to(device)).cpu().numpy().flatten()
        preds = (probs >= 0.5).astype(int)
        wrong = np.where(preds != y_b.numpy())[0]
        for idx in wrong:
            mis_imgs.append(X_b[idx]); mis_true.append(y_b[idx].item()); mis_pred.append(preds[idx])
        if len(mis_imgs) >= 8:
            break

n_show = min(8, len(mis_imgs))
fig, axes = plt.subplots(1, n_show, figsize=(15, 3))
for i in range(n_show):
    img = denormalize(mis_imgs[i]).permute(1,2,0).numpy()[:,:,0]
    axes[i].imshow(img, cmap='gray'); axes[i].axis('off')
    axes[i].set_title(
        f'True: {test_dataset.classes[mis_true[i]]}\nPred: {test_dataset.classes[mis_pred[i]]}',
        fontsize=7, color='red')
plt.suptitle(f'{best_name} — Misclassified Test Images', fontsize=12)
plt.tight_layout(); plt.show()

In [ ]:
# First-layer filters of the Custom CNN
weights = cnn.features[0].weight.data.cpu()  # (32, 3, 3, 3)
filters = weights[:, 0, :, :]
filters = (filters - filters.min()) / (filters.max() - filters.min() + 1e-8)

fig, axes = plt.subplots(4, 8, figsize=(14, 7))
for i, ax in enumerate(axes.flatten()):
    if i < filters.shape[0]:
        ax.imshow(filters[i].numpy(), cmap='gray', interpolation='nearest')
    ax.axis('off')
plt.suptitle('Custom CNN — First Layer Filters (32 kernels, 3x3)', fontsize=12)
plt.tight_layout(); plt.show()

In [ ]:
# Feature maps from Block 1 on a test image
sample_img, sample_label = test_dataset[0]
sample_tensor = sample_img.unsqueeze(0).to(device)

feature_maps = {}
def hook_fn(module, input, output):
    feature_maps['block1'] = output.detach().cpu()

hook = cnn.features[2].register_forward_hook(hook_fn)  # after first ReLU
cnn.eval()
with torch.no_grad():
    _ = cnn(sample_tensor)
hook.remove()

fmaps = feature_maps['block1'][0]  # (32, H, W)
fig, axes = plt.subplots(4, 8, figsize=(14, 7))
for i, ax in enumerate(axes.flatten()):
    if i < fmaps.shape[0]:
        fm = fmaps[i].numpy()
        fm = (fm - fm.min()) / (fm.max() - fm.min() + 1e-8)
        ax.imshow(fm, cmap='viridis')
    ax.axis('off')
plt.suptitle(f'Feature Maps — Block 1 ReLU | Input: {test_dataset.classes[sample_label]}', fontsize=12)
plt.tight_layout(); plt.show()

## 11. Conclusions

- CNNs exploit **spatial locality** and **translation equivariance** via shared convolutional kernels, making them far more parameter-efficient than MLPs on image data.
- **Batch Normalisation** and **Dropout** are essential regularisation tools that stabilise training and reduce overfitting on medical imaging datasets.
- **Transfer learning** (ResNet-18 fine-tuned from ImageNet) consistently outperforms training from scratch, especially when labelled data is limited — a key practical insight for real-world medical imaging.
- **Class imbalance** (more Pneumonia than Normal samples) is addressed with `WeightedRandomSampler`, ensuring the model does not simply predict the majority class.
- Visualising **first-layer filters** and **feature maps** confirms that early layers detect low-level patterns (edges, textures), which compose into higher-level structures in deeper layers.
- The **ROC-AUC** metric is more informative than accuracy alone for imbalanced binary classification tasks like disease detection.
- This notebook completes the progression: perceptron -> MLP -> CNN, showing how architectural choices directly address the limitations of simpler models.